#### Aqbil Gradiansyah
#### 105222138
#### Random Forest
#### Dataset The Soko CTC 2024-2026
#### 

## Import Library yang akan digunakan

In [1]:
# Cell 1 — Import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import export_text
import warnings
import joblib
warnings.filterwarnings('ignore')

In [2]:
# Load & gabung 
df_minuman = pd.read_excel('Dataset/Daily Sales Menu Minuman.xlsx', skiprows=13)
df_makanan = pd.read_excel('Dataset/Daily Sales Menu Makanan.xlsx', skiprows=13)
df = pd.concat([df_minuman, df_makanan], ignore_index=True)
print(df.shape)
df.head()

(19542, 14)


,Branch,Sales Date,Sales Type,Category,Category Detail,Menu Name,Menu Code,Type,Qty,Subtotal,Service Charge,Tax Total,VAT Total,Total
0,- All -,2024-01-01,Sales,DRINK,CHOCOLATE,Choco Java Hot,NaN,Ala Carte,3,84000,4200,8820.0,0,97020.0
1,- All -,2024-01-01,Sales,DRINK,CHOCOLATE,Choco Java Ice,NaN,Ala Carte,3,90000,4500,9450.0,0,103950.0
2,- All -,2024-01-01,Sales,DRINK,COFFE & MANUAL BREW,Hot Americano,NaN,Ala Carte,1,20000,1000,2100.0,0,23100.0
3,- All -,2024-01-01,Sales,DRINK,COFFE & MANUAL BREW,Iced Americano,NaN,Ala Carte,3,66000,3300,6930.0,0,76230.0
4,- All -,2024-01-01,Sales,DRINK,COFFE & MANUAL BREW,Es Kopi Soko,NaN,Ala Carte,9,252000,12600,26460.0,0,291060.0


In [3]:
# Seleksi kolom, parsing, strip whitespace 
df = df[['Sales Date', 'Menu Name', 'Qty', 'Type']]
df['Sales Date'] = pd.to_datetime(df['Sales Date'])
df['Qty'] = pd.to_numeric(df['Qty'], errors='coerce')
df['Menu Name'] = df['Menu Name'].str.strip()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19542 entries, 0 to 19541
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Sales Date  19542 non-null  datetime64[us]
 1   Menu Name   19542 non-null  str           
 2   Qty         19542 non-null  int64         
 3   Type        19542 non-null  str           
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 1.0 MB


In [4]:
# Cell 4 — Cleaning (sama persis)
print("Missing values:\n", df.isnull().sum())
print("Duplicates:", df.duplicated().sum())
print("\nNilai unik kolom Type:\n", df['Type'].value_counts())
df = df.dropna()
df = df.drop_duplicates()
df = df[df['Qty'] > 0]
print("\nSetelah cleaning:", df.shape)

Missing values:
 Sales Date    0
Menu Name     0
Qty           0
Type          0
dtype: int64
Duplicates: 10

Nilai unik kolom Type:
 Type
Ala Carte    19516
Free Item       26
Name: count, dtype: int64

Setelah cleaning: (19532, 4)


In [5]:
# Cell 5 — Drop Type & agregasi harian (sama persis)
df = df.drop(columns=['Type'])
df_agg = df.groupby(['Sales Date', 'Menu Name'])['Qty'].sum().reset_index()
df_agg.columns = ['date', 'menu', 'qty']
print(df_agg.shape)
df_agg.head(10)

(19500, 3)


,date,menu,qty
0,2024-01-01,Aglio olio pasta spaghetti,1
1,2024-01-01,Bolognais Pasta Spaghetti,1
2,2024-01-01,Carbonara pasta spaghetti,1
3,2024-01-01,Charcoal Latte Ice,2
4,2024-01-01,Choco Java Hot,3
5,2024-01-01,Choco Java Ice,3
6,2024-01-01,Choco lava,1
7,2024-01-01,Chocolatte banana roll,1
8,2024-01-01,Cireng Bumbu,4
9,2024-01-01,Es Kopi Soko,9


In [6]:
# Cell 6 — Date gap filling (sama persis)
all_dates = pd.date_range(df_agg['date'].min(), df_agg['date'].max())
all_menus = df_agg['menu'].unique()
full_index = pd.MultiIndex.from_product([all_dates, all_menus], names=['date', 'menu'])
df_full = df_agg.set_index(['date', 'menu']).reindex(full_index, fill_value=0).reset_index()
print(df_full.shape)
df_full.head(10)

(132492, 3)


,date,menu,qty
0,2024-01-01,Aglio olio pasta spaghetti,1
1,2024-01-01,Bolognais Pasta Spaghetti,1
2,2024-01-01,Carbonara pasta spaghetti,1
3,2024-01-01,Charcoal Latte Ice,2
4,2024-01-01,Choco Java Hot,3
5,2024-01-01,Choco Java Ice,3
6,2024-01-01,Choco lava,1
7,2024-01-01,Chocolatte banana roll,1
8,2024-01-01,Cireng Bumbu,4
9,2024-01-01,Es Kopi Soko,9


In [7]:
# Cell 7 — Feature engineering (sama persis)
df_full = df_full.sort_values(['menu', 'date']).reset_index(drop=True)
df_full['day_of_week']   = df_full['date'].dt.dayofweek
df_full['month']         = df_full['date'].dt.month
df_full['is_weekend']    = df_full['day_of_week'].isin([5, 6]).astype(int)
df_full['week_of_month'] = df_full['date'].dt.day // 7 + 1
df_full['lag_1']         = df_full.groupby('menu')['qty'].shift(1)
df_full['lag_3']         = df_full.groupby('menu')['qty'].shift(3)
df_full['lag_7']         = df_full.groupby('menu')['qty'].shift(7)
df_full['lag_14']        = df_full.groupby('menu')['qty'].shift(14)
df_full['rolling_7']     = df_full.groupby('menu')['qty'].transform(
    lambda x: x.shift(1).rolling(7).mean()
)
df_full['rolling_14']    = df_full.groupby('menu')['qty'].transform(
    lambda x: x.shift(1).rolling(14).mean()
)
df_full['rolling_std_7'] = df_full.groupby('menu')['qty'].transform(
    lambda x: x.shift(1).rolling(7).std()
)
df_full = df_full.dropna()
print(df_full.shape)
df_full.head(10)

(129958, 14)


,date,menu,qty,day_of_week,month,is_weekend,week_of_month,lag_1,lag_3,lag_7,lag_14,rolling_7,rolling_14,rolling_std_7
14,2024-01-15,Aeropress Import,0,0,1,0,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15,2024-01-16,Aeropress Import,0,1,1,0,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16,2024-01-17,Aeropress Import,0,2,1,0,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17,2024-01-18,Aeropress Import,0,3,1,0,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0
18,2024-01-19,Aeropress Import,0,4,1,0,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19,2024-01-20,Aeropress Import,0,5,1,1,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20,2024-01-21,Aeropress Import,0,6,1,1,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
21,2024-01-22,Aeropress Import,0,0,1,0,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22,2024-01-23,Aeropress Import,0,1,1,0,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
23,2024-01-24,Aeropress Import,0,2,1,0,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
# Cell 8 — Split train/test 80:20 time-based (sama persis)
df_full = df_full.sort_values('date').reset_index(drop=True)
split_idx = int(len(df_full) * 0.8)
split_date = df_full.iloc[split_idx]['date']
print("Split date:", split_date)
train = df_full[df_full['date'] < split_date]
test  = df_full[df_full['date'] >= split_date]
print("Train:", train.shape, "| Test:", test.shape)

Split date: 2025-08-11 00:00:00
Train: (103894, 14) | Test: (26064, 14)


In [9]:
# Cell 9 — Features
features = ['day_of_week', 'month', 'is_weekend', 'week_of_month',
            'lag_1', 'lag_3', 'lag_7', 'lag_14',
            'rolling_7', 'rolling_14', 'rolling_std_7']

X_train = train[features]
y_train = train['qty']
X_test  = test[features]
y_test  = test['qty']

In [10]:
# v1 — RF Baseline (n=100, default)
rf_v1 = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)
rf_v1.fit(X_train, y_train)
y_pred_rf_v1 = rf_v1.predict(X_test)

print("RF v1 - Baseline (n=100, default)")
print(f"MAE  : {mean_absolute_error(y_test, y_pred_rf_v1):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_rf_v1)):.4f}")
print(f"R²   : {r2_score(y_test, y_pred_rf_v1):.4f}")

RF v1 - Baseline (n=100, default)
MAE  : 0.4117
RMSE : 0.9531
R²   : 0.3942


In [11]:
# RF Tuned (n=200, max_depth=10)
rf_v2 = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
rf_v2.fit(X_train, y_train)
y_pred_rf = rf_v2.predict(X_test)
print("RF v2 - Tuned (n=200, depth=10)")
print(f"MAE  : {mean_absolute_error(y_test, y_pred_rf):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.4f}")
print(f"R²   : {r2_score(y_test, y_pred_rf):.4f}")

RF v2 - Tuned (n=200, depth=10)
MAE  : 0.3793
RMSE : 0.9068
R²   : 0.4516


In [12]:
# v3 — RF Tuned (n=300, max_depth=15)
rf_v3 = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
rf_v3.fit(X_train, y_train)
y_pred_rf_v3 = rf_v3.predict(X_test)

print("RF v3 - Tuned (n=300, depth=15)")
print(f"MAE  : {mean_absolute_error(y_test, y_pred_rf_v3):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_rf_v3)):.4f}")
print(f"R²   : {r2_score(y_test, y_pred_rf_v3):.4f}")

RF v3 - Tuned (n=300, depth=15)
MAE  : 0.3892
RMSE : 0.9210
R²   : 0.4343


In [15]:
# Tabel perbandingan 3 versi RF
summary_rf = pd.DataFrame({
    'Versi': [
        'RF v1 - Baseline (n=100, default)',
        'RF v2 - Tuned (n=200, depth=10)',
        'RF v3 - Tuned (n=300, depth=15)'
    ],
    'MAE': [
        mean_absolute_error(y_test, y_pred_rf_v1),
        mean_absolute_error(y_test, y_pred_rf),
        mean_absolute_error(y_test, y_pred_rf_v3)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, y_pred_rf_v1)),
        np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        np.sqrt(mean_squared_error(y_test, y_pred_rf_v3))
    ],
    'R²': [
        r2_score(y_test, y_pred_rf_v1),
        r2_score(y_test, y_pred_rf),
        r2_score(y_test, y_pred_rf_v3)
    ]
})

print(summary_rf.to_string(index=False))

                            Versi      MAE     RMSE       R²
RF v1 - Baseline (n=100, default) 0.409650 0.947681 0.394796
  RF v2 - Tuned (n=200, depth=10) 0.377208 0.902074 0.451644
  RF v3 - Tuned (n=300, depth=15) 0.386641 0.917581 0.432629


In [14]:
from sklearn.tree import export_text
import pandas as pd

# Define ulang semua yang dibutuhkan
feature_names = features  # nama variabel di kode kamu adalah 'features'

# Cari sampel yang aktualnya > 0
idx_bagus = y_test[y_test > 0].index[5]
sampel_rf = X_test.loc[[idx_bagus]]
aktual_rf = y_test.loc[idx_bagus]

# STRUKTUR POHON PERTAMA
satu_pohon = rf_v2.estimators_[0]
print("STRUKTUR POHON PERTAMA (max depth 4)")
print(export_text(satu_pohon, feature_names=feature_names, max_depth=4))

# PREDIKSI TIAP POHON
prediksi_tiap_pohon = [tree.predict(sampel_rf)[0] for tree in rf_v2.estimators_]
rata_rata = sum(prediksi_tiap_pohon) / len(prediksi_tiap_pohon)

print("\nSAMPEL PREDIKSI PER POHON")
tabel_pohon = pd.DataFrame({
    'Pohon': [f'Pohon #{i+1}' for i in range(5)] + ['...', 'Pohon #200'],
    'Prediksi': [round(p, 4) for p in prediksi_tiap_pohon[:5]] + ['...', round(prediksi_tiap_pohon[-1], 4)]
})
print(tabel_pohon.to_string(index=False))
print(f"\nRata-rata 200 pohon   : {rata_rata:.4f}")
print(f"Prediksi model RF     : {rf_v2.predict(sampel_rf)[0]:.4f}")
print(f"Aktual                : {aktual_rf}")
print(f"Error (|Aktual - Ŷ|)  : {abs(aktual_rf - rata_rata):.4f}")

STRUKTUR POHON PERTAMA (max depth 4)
|--- rolling_14 <= 3.54
|   |--- rolling_14 <= 0.39
|   |   |--- rolling_14 <= 0.11
|   |   |   |--- rolling_14 <= 0.04
|   |   |   |   |--- day_of_week <= 4.50
|   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |   |--- day_of_week >  4.50
|   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |--- rolling_14 >  0.04
|   |   |   |   |--- day_of_week <= 3.50
|   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |   |--- day_of_week >  3.50
|   |   |   |   |   |--- truncated branch of depth 6
|   |   |--- rolling_14 >  0.11
|   |   |   |--- day_of_week <= 4.50
|   |   |   |   |--- rolling_std_7 <= 0.43
|   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |   |--- rolling_std_7 >  0.43
|   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |--- day_of_week >  4.50
|   |   |   |   |--- rolling_14 <= 0.25
|   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |   |--- rolling_14 >  0.25


In [17]:
print(f"Index sampel  : {idx_bagus}")
print(f"Tanggal       : {df_full.loc[idx_bagus, 'date']}")
print(f"Menu          : {df_full.loc[idx_bagus, 'menu']}")
print(f"Aktual Qty    : {df_full.loc[idx_bagus, 'qty']}")
print(f"\nNilai fitur:")
print(X_test.loc[idx_bagus])

Index sampel  : 105171
Tanggal       : 2025-08-11 00:00:00
Menu          : Cireng Bumbu
Aktual Qty    : 1

Nilai fitur:
day_of_week      0.000000
month            8.000000
is_weekend       0.000000
week_of_month    2.000000
lag_1            1.000000
lag_3            2.000000
lag_7            0.000000
lag_14           1.000000
rolling_7        1.000000
rolling_14       1.000000
rolling_std_7    1.527525
Name: 105171, dtype: float64


### Export Model hasil training 

In [20]:
import joblib

# Simpan model rf_v2 ke dalam file bernama 'model_rf_v2.pkl'
joblib.dump(rf_v2, "model_rf_v2.pkl")

print("Model Random Forest V2 berhasil disimpan!")

Model Random Forest V2 berhasil disimpan!


### Export Dataframe yang sudah ada Rekayasa Fitur (df_full)

In [13]:
results_rf = test[['date', 'menu']].copy()
results_rf['y_actual'] = y_test.values
results_rf['y_pred_rf'] = y_pred_rf

#df_full.to_csv('df_full.csv', index=False)
results_rf.head(10)

,date,menu,y_actual,y_pred_rf
103894,2025-08-11,Extra Espresso,0,0.136819
103895,2025-08-11,Baby Purple,0,0.003194
103896,2025-08-11,Iced Vanilla Latte,0,0.003194
103897,2025-08-11,Iga Bakar,0,0.003194
103898,2025-08-11,Sate Jando,0,0.427438
103899,2025-08-11,Buntut Bakar,0,0.003194
103900,2025-08-11,Donat,0,0.003194
103901,2025-08-11,Es Kopi Soko,2,3.719011
103902,2025-08-11,Brulle Creamy Macaroni,0,0.232010
103903,2025-08-11,Crispy Chicken Dice,0,0.429510


## Eksplosi BOM

In [13]:
file = 'Dataset/SOKO - Master Menu Soko.xlsx'

# 1. LOAD MENU BOM
bom_bar     = pd.read_excel(file, sheet_name='Menu Bar', header=4)
bom_kitchen = pd.read_excel(file, sheet_name='Menu Kitchen', header=4)

bom_all = pd.concat([bom_bar, bom_kitchen], ignore_index=True)

bom_all.columns = bom_all.columns.str.strip()
bom_all = bom_all.dropna(how='all')

# forward fill menu
bom_all['Menu'] = bom_all['Menu'].ffill()

# cleaning
bom_all['Menu']  = bom_all['Menu'].astype(str).str.strip().str.lower()
bom_all['Bahan'] = bom_all['Bahan'].astype(str).str.strip().str.lower()

bom_all = bom_all.dropna(subset=['Bahan'])

bom_all['Recipe QTY'] = pd.to_numeric(bom_all['Recipe QTY'], errors='coerce').fillna(0)

bom_all = bom_all.rename(columns={
    'Menu': 'menu',
    'Bahan': 'bahan',
    'Recipe QTY': 'recipe_qty',
    'Satuan': 'satuan'
})

bom_all = bom_all[['menu','bahan','recipe_qty','satuan']]

# 2. LOAD PRODUCTION BOM (KTC)
prod_bar = pd.read_excel(file, sheet_name='Production Bar', header=4)
prod_kitchen = pd.read_excel(file, sheet_name='Production Kitchen', header=4)

prod_all = pd.concat([prod_bar, prod_kitchen], ignore_index=True)

prod_all.columns = prod_all.columns.str.strip()
prod_all = prod_all.dropna(how='all')

# forward fill nama KTC
prod_all['Menu'] = prod_all['Menu'].ffill()

# tandai baris yield (yang child kosong)
prod_all['is_yield'] = prod_all['Bahan'].isna()

# cleaning
prod_all['Menu']  = prod_all['Menu'].astype(str).str.strip().str.lower()
prod_all['Bahan'] = prod_all['Bahan'].astype(str).str.strip().str.lower()

prod_all['Recipe QTY'] = pd.to_numeric(prod_all['Recipe QTY'], errors='coerce').fillna(0)

# 2A. HITUNG YIELD KTC
yield_df = prod_all[prod_all['is_yield']].copy()
yield_df = yield_df.rename(columns={
    'Menu': 'parent',
    'Recipe QTY': 'yield_qty'
})[['parent','yield_qty']]

# 2B. DETAIL BAHAN KTC
prod_detail = prod_all[~prod_all['is_yield']].copy()

prod_detail = prod_detail.rename(columns={
    'Menu': 'parent',
    'Bahan': 'child',
    'Recipe QTY': 'qty_prod',
    'Satuan': 'satuan_prod'
})

prod_detail = prod_detail[['parent','child','qty_prod','satuan_prod']]

# 2C. NORMALISASI (PAKAI YIELD)
prod_detail = prod_detail.merge(yield_df, on='parent', how='left')

# proporsi bahan per 1 unit KTC
prod_detail['qty_per_unit'] = prod_detail['qty_prod'] / prod_detail['yield_qty']

# 3. PREDIKSI RF
pred = results_rf[['date','menu','y_pred_rf']].copy()
pred['menu'] = pred['menu'].str.lower().str.strip()
pred['y_pred_rf'] = pred['y_pred_rf'].clip(lower=0).round()

# 4. LEVEL 1: MENU → KTC
menu_join = pred.merge(bom_all, on='menu', how='left')

# 5. LEVEL 2: KTC → RAW
full_bom = menu_join.merge(
    prod_detail,
    left_on='bahan',
    right_on='parent',
    how='left'
)

# 6. FINAL CALC
# kalau tidak ada KTC → langsung bahan
full_bom['final_bahan'] = full_bom['child'].fillna(full_bom['bahan'])

# hitung kebutuhan bahan mentah
full_bom['final_qty'] = (
    full_bom['y_pred_rf'] *
    full_bom['recipe_qty'] *
    full_bom['qty_per_unit'].fillna(1)
)

full_bom['final_satuan'] = full_bom['satuan_prod'].fillna(full_bom['satuan'])

# 7. AGREGASI BAHAN
stok_harian_rf = full_bom.groupby(
    ['date','final_bahan','final_satuan']
)['final_qty'].sum().reset_index()

stok_harian_rf.columns = ['Tanggal','Bahan','Satuan','Kebutuhan']
stok_harian_rf['Kebutuhan'] = stok_harian_rf['Kebutuhan'].round(2)

# 8. OPTIONAL: AGREGASI KTC
ktc_usage = menu_join.groupby(
    ['date','bahan','satuan']
).apply(lambda x: (x['y_pred_rf'] * x['recipe_qty']).sum()).reset_index(name='kebutuhan_ktc')

# 9. OUTPUT
stok_harian_rf.to_csv('rekomendasi_stok_harian_rf.csv', index=False)

print("BAHAN MENTAH")
print(stok_harian_rf.head(10))

print("\nKEBUTUHAN KTC")
print(ktc_usage.head(10))

# LABEL KTC
ktc_usage_renamed = ktc_usage.rename(columns={
    'bahan': 'Item',
    'satuan': 'Satuan',
    'kebutuhan_ktc': 'Kebutuhan'
})

ktc_usage_renamed['Tipe'] = 'KTC'

# LABEL RAW MATERIAL
raw_usage = stok_harian_rf.rename(columns={
    'Bahan': 'Item',
    'Satuan': 'Satuan',
    'Kebutuhan': 'Kebutuhan'
})

raw_usage['Tipe'] = 'RAW'

# GABUNGKAN
combined_usage = pd.concat([
    ktc_usage_renamed,
    raw_usage
], ignore_index=True)

# TOTAL PER ITEM
total_usage = combined_usage.groupby(
    ['Tanggal','Item','Satuan','Tipe']
)['Kebutuhan'].sum().reset_index()

total_usage = total_usage.sort_values(
    by=['Tanggal','Kebutuhan'],
    ascending=[True, False]
)
#urutkan ulang kolom
total_usage = total_usage[['Tanggal','Item','Tipe','Kebutuhan','Satuan']]

# OUTPUT
print("TOTAL GABUNGAN KTC + RAW")
print(total_usage.head(20))

# FILTER PER TANGGAL (CONTOH)
tanggal_filter = '2025-08-11'

print(f"\n=== DETAIL {tanggal_filter} ===")
print(
    total_usage[total_usage['Tanggal'] == tanggal_filter]
    .sort_values(by='Kebutuhan', ascending=False)
    .head(20)
)

# 10. DEBUG PER MENU (DETAIL)
full_bom[
    (full_bom['date'] == '2025-08-11') & 
    (full_bom['menu'] == 'naga black tea ice') &
    (full_bom['final_qty'] > 0)
][['menu','y_pred_rf','bahan','final_bahan','final_qty','final_satuan']] \
.sort_values(by='final_qty', ascending=False)

BAHAN MENTAH
     Tanggal                     Bahan Satuan  Kebutuhan
0 2025-08-11                 air galon     ml     590.91
1 2025-08-11   air mineral water 600ml  Botol       1.00
2 2025-08-11  arghani chocolate mixing     gr      30.00
3 2025-08-11         asam jawa kemasan     gr       0.10
4 2025-08-11        ayam boneless dada     gr      60.24
5 2025-08-11                ayam sayap     gr       0.00
6 2025-08-11             baking powder     gr       0.00
7 2025-08-11             bakso mas eko    pcs       0.00
8 2025-08-11             bawang bombay     gr      31.30
9 2025-08-11              bawang merah     gr       0.04

KEBUTUHAN KTC
        date                     bahan satuan  kebutuhan_ktc
0 2025-08-11                 air galon     ml          297.0
1 2025-08-11   air mineral water 600ml  Botol            1.0
2 2025-08-11  arghani chocolate mixing     gr           30.0
3 2025-08-11        ayam boneless dada     gr           60.0
4 2025-08-11             bakso mas eko  

,menu,y_pred_rf,bahan,final_bahan,final_qty,final_satuan
1129,naga black tea ice,1.0,bar pro black tea,air galon,215.264188,ml
1130,naga black tea ice,1.0,bar pro simple syrup,gula hijau,45.000000,gr
1131,naga black tea ice,1.0,bar pro simple syrup,air galon,45.000000,ml
1128,naga black tea ice,1.0,bar pro black tea,teh naga,4.735812,gr
1132,naga black tea ice,1.0,cup oval 12 oz custom soko,cup oval 12 oz custom soko,1.000000,pcs
